In [1]:
#!pip install google-genai
#!pip install "tensorflow<2.22,>=2.21"
#!pip install google-generativeai

#!pip install parse


In [2]:
# Setup debugging and connection parameters
CONNECT_TO_CHATGPT_API = True

CONNECT_TO_GEMINI_API = True
GEMINI_MODEL_NAME = "gemini-3.6-flash"
GEMINI_TEMPERATURE = 0.9
GEMINI_TOP_P = 0.95
GEMINI_TOP_K = 40
GEMINI_MAX_TOKENS = 2048
GEMINI_TIMEOUT = 30

REASONING_THRESHOLD = 0.70
EVALUATION_DEBUG_LEVEL = 2
DEBUG_LEVEL = EVALUATION_DEBUG_LEVEL
VAGUE_WRONG_ANSWER = "dunno"
VAGUE_WRONG_REASONING = "dunno"

CONTROL_RUN = False


In [3]:
# Note:  In terms of "scaling data," adding the following related problem and related properties,
#        to math problem 2 below, raised the cosine similarity of a student response
#        "4 * 250 = 4 * (25 * 10) = (4 * 25) * 10 = 100 * 10 = 1000" to response content wanted by the teacher
#        from around 41% to around 85%.
#
#   ...
#   [
#   ...
#   { "problem":  "4 * 250",
#     "answer":  "1000",
#     related_problems":  [
#     ...
#       { "problem":  "25 * 10",
#         "answer": 250
#       }
#     ],
#     "related_properties": [ { "property":  "associative",
#                               "usage":  "4 * (25 * 10) = (4 * 25) * 10"
#                             }
#                           ]
#   }
#   ...
#   ]

# 1.  Read a list of math arithmetic problems from a JSON file
#
# For example:
# [
# {
#   "num_candidate_answers":  10,
#   "retry_max_times": 10,
#   "problems":  [
#   { "problem":  "4 * 25",
#     "answer":  "100",
#     "related_problems":  [ { "problem":  "How many quarters are in a dollar",
#                              "answer":  4
#                            }
#                          ]
#   },
#   { "problem":  "4 * 250",
#     "answer":  "1000",
#     "related_problems":  [ { "problem":  "4 * 25",
#                              "answer":  100
#                            }
#                          ]
#   },
#   { "problem":  "1000 / 4",
#     "answer":  "250",
#     "related_problems":  [ { "problem":  "4 * 250",
#                              "answer":  1000
#                            }
#                          ]
#   },
#   { "problem":  "996 / 4",
#     "answer":  "249",
#     "related_problems":  [ { "problem":  "1000 / 4",
#                              "answer":  250
#                            }
#                          ],
#     "related_properties": [ { "property":  "distributive",
#                               "usage":  "996 / 4 = (1000 - 4 ) / 4 = (1000 + (-4)) * (1/4) = (1000 * (1/4)) + ((-4) * (1/4)) = (1000 / 4) + (-4/4) = 250 - 1 = 249"
#                             }
#                           ],
#     "common_errors": [ { "error":  "996 / 4 = 6",
#                          "reasoning":  "9 + 9 + 6 = 24, 24 / 4 = 6" }
#                      ]
#   }
#   ]
# }
# ]


In [ ]:
# 2.  Connect to the ChatGPT API and Gemini API

import os
from openai import OpenAI
import google.generativeai as genai

if (CONNECT_TO_CHATGPT_API):
  client = OpenAI(
      api_key=os.environ.get("OPENAI_API_KEY"),
  )

  # Briefly test the connection
  response = client.responses.create(
      model="gpt-5.5",
      instructions="You are a math teacher assistant.",
      input="What is 2 + 2?"
  )

  if (DEBUG_LEVEL > 0):
    print("OpenAI API basic test response:  %s" % response.output_text)

if (CONNECT_TO_GEMINI_API):      
  if (DEBUG_LEVEL > 0):
    print("About to connect to Gemini.")

  gemini_api_key = os.getenv("GEMINI_API_KEY")

  if (DEBUG_LEVEL > 0):
    print("GEMINI_API_KEY is:  %s" % gemini_api_key)

  try:
    genai.configure(api_key=gemini_api_key)

    if (DEBUG_LEVEL > 0):
      print("Gemini connect point 1")
        
    model = genai.GenerativeModel(
                model_name=GEMINI_MODEL_NAME,
                generation_config={
                  "temperature": GEMINI_TEMPERATURE,
                  "top_p": GEMINI_TOP_P,
                  "top_k": GEMINI_TOP_K,
                  "max_output_tokens": GEMINI_MAX_TOKENS,
                },
                safety_settings=[]
      )

    if (DEBUG_LEVEL > 0):
      print("Gemini connect point 2")

    gemini_session = model.start_chat(history=[])

  except Exception as e:
    print("ERROR:  Gemini got an error:  %s" % str(e))


In [5]:
# 3.  Iteratively looping over the math problems read from step 1, do the following steps:
#   (a)  Ask a student the math problem from the current list iteration.  For example:  996 / 4.
#   (b)  Pass the student's answer to ChatGPT and ask ChatGPT to verify if the student is correct (correct answers can
#        come in different forms, and this allows ChatGPT to resolve whether the student has provided an equivalent
#        correct answer in a different form.  For instance, 996 / 4 = 249 but it also equals "249.0 and 0/4".)
#   (c)  Whether the student was correct or incorrect in step 3(b), ask the student for what reasoning went into the answer.
#        If the answer came in a format unexpected for the question (for instance if a decimal number answer was provided when
#        a fraction was expected), ask the student why they chose a particular format for the answer.
#      (i)  If the student's answer was correct and also contains elements expected in reasoning (determined by cosine
#           similarity between the student's answer and reasoning provided for the problem in the JSON problem inputs), break
#           and go into the next loop iteration (for the next math problem in the list).
#      (ii)  If the student's answer was incorrect and/or did not contain reasoning elements expected for
#            an answer to the problem, prepare text melding the student's answer with input text from the JSON file
#            and ask ChatCPT to formulate a set of productive questions to help lead the student in a good direction.
#            Specify ask ChatGPT to provide n candidate answers to pick from, where n was specified in the JSON setup
#            file read in step 1 above.  For example:
#            “A child incorrectly thinks that 996 / 4 = 6
#             when correctly 996 / 4 = (1000 - 4 ) / 4 = (1000 + (-4)) * (1/4)
#             = (1000 * (1/4)) + ((-4) * (1/4)) = (1000 / 4) + (-4/4) = 250 - 1 = 249,
#             using the distributive property.  Ask a question that asks the child a
#             productive question (not an answer) containing correct reasoning.  Provide 10 possible
#             productive questions.  The child might have incorrectly reasoned that
#             996 / 4 = 6 because 9 + 9 + 6 = 24 and 24 / 4 = 6.”
#      (iii)  Read the ChatGPT answers and pick the "best" one to convey back to the student, using TF-IDF and cosine similarity
#             between notes in the JSON input file, text originating from the student, and the answers provided by ChatGPT.
#      (iv)  Continue steps (c)(i) through (c)(iii) until the student gives a correct answer containing at least some elements
#            of reasoning specified as desired in the JSON input file, or if the student wants to give up on the problem
#            (the correct answer should not be provided) or if a certain number of attempts t, read in from the JSON file
#             in step 1, was exceeded).

In [6]:
def get_text_attributes_from_problem(problem_df):
  text_attributes_from_problem = ""

  if ('related_properties' in problem_df.columns):
    related_properties_df = pd.DataFrame.from_dict(problem_df['related_properties']).dropna()
    if (len(related_properties_df) > 0):
      for index in range(0, len(related_properties_df)):
        property_text = "; %s:  %s" %  (related_properties_df['related_properties'][index]['property'], related_properties_df['related_properties'][index]['usage'])
        text_attributes_from_problem += property_text

  if ('related_problems' in problem_df.columns):
    related_problems_df = pd.DataFrame.from_dict(problem_df['related_problems']).dropna()
    if (len(related_problems_df) > 0):
      for index in range(0, len(related_problems_df)):
        property_text = "; %s = %s" %  (related_problems_df['related_problems'][index]['problem'], related_problems_df['related_problems'][index]['answer'])
        text_attributes_from_problem += property_text

  return text_attributes_from_problem


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def get_content_similarity(input_text, texts_to_correlate):
  # This function uses TF-IDF and cosine similarity

  # Note:  This function uses just TF-IDF and cosine similarity to characterize how relevant
  #        student reasoning is to a math problem.
  #
  #        As previously discussed with my UMGC AI bootcamp mentor, interactions with the
  #        ChatGPT API are more than sufficient for implementing my math chatbot capstone project.
  #
  #        But it is possible that this get_content_similarity() function can be improved by
  #        exploring classifiers or regressors in scikit-learn, or deep neural networks in keras,
  #        to see if inputs of student reasoning are really relevant to a math problem.
  #        Training data for such a model could be the fields "question" and "conversation" (for X)
  #        and "self-correctness" (for y) in the math conversation dataset at:
  #        https://github.com/eth-nlped/mathdial

  texts_list = [ input_text ]
  texts_list.extend(texts_to_correlate)

  if (DEBUG_LEVEL > 0):
    print("input_text:  %s, texts_to_correlate:  %s, texts_list:  %s" % (input_text, texts_to_correlate, texts_list))

  vectorizer = TfidfVectorizer(min_df=2, max_df=0.7)
  vectorizer = TfidfVectorizer()
  vectorized_data = vectorizer.fit_transform(texts_list)

  tfidf_df = pd.DataFrame(vectorized_data.toarray(),
                          columns=vectorizer.get_feature_names_out())
  tfidf_df.index = texts_list

  # Compute the cosine similarity
  cosine_similarity_array = cosine_similarity(tfidf_df)

  # Get the indices of similar texts based on cosine similarity
  cosine_similarity_df = pd.DataFrame(cosine_similarity_array, index=tfidf_df.index, columns=tfidf_df.index)

  return cosine_similarity_df


In [8]:
def student_reasoning_is_relevant_to_problem(text_attributes_from_problem, student_reasoning, previous_student_reasoning):
  # Find similarities between student reasoning (and a list of previous student reasoning) and
  # teacher notes in the text_attributes_from_problem (that were read from the problem set .JSON file).

  if (DEBUG_LEVEL > 0):
    print("student_reasoning:  %s" % student_reasoning)
    print("text_attributes_from_problem:  %s" % text_attributes_from_problem)

  content_similarity_df = get_content_similarity(student_reasoning, [ text_attributes_from_problem ])
  student_reasoning_similarity_series = content_similarity_df[student_reasoning][1:]

  if (DEBUG_LEVEL > 0):
    print("student_reasoning_similarity_series:  %s" % str(student_reasoning_similarity_series))
    print("student_reasoning_similarity_series.max().size:  %s" % str(student_reasoning_similarity_series.max().size))
    print("student_reasoning_similarity_series.max().shape:  %s" % str(student_reasoning_similarity_series.max().shape))

  student_reasoning_similarity_percentage = student_reasoning_similarity_series.max()
  if (student_reasoning_similarity_series.max().size > 1):   # Remove max duplicates if necessary
    if (DEBUG_LEVEL > 0):
      print("student_reasoning_similarity_series.max().iloc[0]:  %s" % str(student_reasoning_similarity_series.max().iloc[0]))
    student_reasoning_similarity_percentage = student_reasoning_similarity_series.max().iloc[0]

  return student_reasoning_similarity_percentage >= REASONING_THRESHOLD


In [9]:
def stub_to_correlate_phrases_in_a_0_to_10_ranking(text_phrase_0, text_phrase_1):

  #return 10
  #return 0

  content_similarity_df = get_content_similarity(text_phrase_0, [ text_phrase_1 ])
  reasoning_similarity_series = content_similarity_df[text_phrase_0][1:]

  if (DEBUG_LEVEL > 0):
    print("reasoning_similarity_series:  %s" % str(reasoning_similarity_series))
    print("reasoning_similarity_series.max().size:  %s" % str(reasoning_similarity_series.max().size))
    print("reasoning_similarity_series.max().shape:  %s" % str(reasoning_similarity_series.max().shape))

  reasoning_similarity_percentage = reasoning_similarity_series.max()
  if (reasoning_similarity_series.max().size > 1):   # Remove max duplicates if necessary
    if (DEBUG_LEVEL > 0):
      print("reasoning_similarity_series.max().iloc[0]:  %s" % str(reasoning_similarity_series.max().iloc[0]))
    reasoning_similarity_percentage = reasoning_similarity_series.max().iloc[0]

  if (DEBUG_LEVEL > 0):
    print("text_phrase_0:  %s" % text_phrase_0)
    print("text_phrase_1:  %s" % text_phrase_1)
    print("reasoning_similarity_percentage is:  %f" % reasoning_similarity_percentage)
      
  return reasoning_similarity_percentage*10.0

from parse import parse

def ask_gemini_to_correlate_phrases_in_a_0_to_10_ranking(text_phrase_0, text_phrase_1):

  if (text_phrase_0 == "" or text_phrase_1 == ""):
    return 5 # If either text_phrase is just empty, return a median correlation rating 5 (out of 10), basically signifying no knowledge either way

  gemini_response = stub_to_correlate_phrases_in_a_0_to_10_ranking(text_phrase_0, text_phrase_1)

  if (CONNECT_TO_GEMINI_API):
    try:
      if (DEBUG_LEVEL > 0):
        print("Gemini debug point 3")

      message = "Rate on a scale of 0 to 10 the relevance of ideas contained in phrase \"" + text_phrase_0 + "\" to ideas in phrase \"" + text_phrase_1 + ".\"  Answer only with one integer number."

      if (DEBUG_LEVEL > 0):
        print("Gemini debug point 4")

      gemini_raw_response = gemini_session.send_message(message)

      if (DEBUG_LEVEL > 0):
        print("Gemini debug point 5")
        print("Gemini response rating for (%s, %s) was:  %s"  % (text_phrase_0, text_phrase_1, gemini_response))

      gemini_text_response = gemini_raw_response.candidates[0].content.parts[0].text

      if (DEBUG_LEVEL > 0):
        print("Gemini debug point 6")
        print("Gemini response text was:  %s"  % (gemini_text_response))

      result = parse("{:d}", gemini_text_response)

      if (DEBUG_LEVEL > 0):
        print("Gemini debug point 7")
        print("Gemini result was:  %s"  % (result))
        gemini_response = result[0]
        print("Gemini rating was:  %s"  % (gemini_response))

    except Exception as e:
      print("ERROR:  Gemini got an error:  %s" % str(e))
      return 0

  return gemini_response


In [10]:
def pick_best_chatgpt_feedback(problem_df, student_reasoning, chatgpt_feedback_list, n):
  # Find similarities etween student reasoning + teacher notes, against a list
  # of chatgpt feedback n items long.

  input_baseline_text = student_reasoning 
  input_baseline_text += get_text_attributes_from_problem(problem_df)

  if (DEBUG_LEVEL > 0):
    print("input_baseline_text:  %s" % input_baseline_text)
    print("chatgpt_feedback_list:  %s" % chatgpt_feedback_list)

  content_similarity_df = get_content_similarity(input_baseline_text, chatgpt_feedback_list)
  reasoning_similarity_series = content_similarity_df.iloc[0, 1:]

  if (DEBUG_LEVEL > 0):
    print("reasoning_similarity_series:  %s" % reasoning_similarity_series)
    print("reasoning_similarity_series.max():  %s" % reasoning_similarity_series.max())
    print("reasoning_similarity_series.idxmax():  %s" % reasoning_similarity_series.idxmax())

  return reasoning_similarity_series.idxmax()


In [11]:
import re

def chatgpt_api_supplies_leading_question(problem_df, student_answer, student_reasoning, num_leading_questions_to_supply):

  leading_question_prompt = "A child incorrectly thinks that %s = %s when correctly %s = %s" \
                            % (problem_df['problem'][0], student_answer, problem_df['problem'][0], problem_df['answer'][0])

  if ('related_properties' in problem_df.columns):
    related_properties_df = pd.DataFrame.from_dict(problem_df['related_properties']).dropna()
    if (len(related_properties_df) > 0):
      related_property = related_properties_df['related_properties'][0]
      related_property_property = related_property['property']
      related_property_usage = related_property['usage']
      leading_question_prompt += " because %s from the %s property" % (related_property_usage, related_property_property)

  leading_question_prompt += ".  Ask the child %s candidate productive questions (not answers) containing correct reasoning.  " \
                             % num_leading_questions_to_supply

  leading_question_prompt += "  The child reasoned that '%s'" % student_reasoning

  if ('common_errors' in problem_df.columns):
    common_errors_df = pd.DataFrame.from_dict(problem_df['common_errors']).dropna()
    if (len(common_errors_df) > 0):
      leading_question_prompt += ", and might have also incorrectly reasoned that: "
      for index in range(0, len(common_errors_df), 1):
        common_error = common_errors_df['common_errors'][index]
        common_error_error = common_error['error']
        common_error_reasoning = common_error['reasoning']
        if (index > 0):
         leading_question_prompt += ", and "
        leading_question_prompt += "  %s" % common_error_reasoning 
      leading_question_prompt += ".  "

  if ('related_problems' in problem_df.columns):
    related_problems_df = pd.DataFrame.from_dict(problem_df['related_problems']).dropna()
    if (len(related_problems_df) > 0):
      leading_question_prompt += "  Related problems:  "
      for index in range(0, len(related_problems_df), 1):
        leading_question_addendum = "%s = %s" %  (related_problems_df['related_problems'][index]['problem'], related_problems_df['related_problems'][index]['answer'])
        leading_question_prompt += leading_question_addendum
        leading_question_prompt += ".  "

  if ('related_properties' in problem_df.columns):
    related_properties_df = pd.DataFrame.from_dict(problem_df['related_properties']).dropna()
    if (len(related_properties_df) > 1):
      leading_question_prompt += "  Other related properties:  "
      for index in range(1, len(related_properties_df), 1):
        leading_question_addendum = "%s due to the %s property" %  (related_properties_df['related_properties'][index]['property'], related_properties_df['related_properties'][index]['usage'])
        leading_question_prompt += leading_question_addendum
        leading_question_prompt += ".  "

  if (DEBUG_LEVEL > 0):
    print("Question giving to ChatGPT:  %s" % leading_question_prompt)

  chatgpt_feedback_list = []
  response_output = "1. If (996 div 4) meant 4 equal groups, would each group having 6 make a total of (4 times 6)? Is that total close to 996?" \
                    "2. Since (1000 div 4 = 250), and 996 is 4 less than 1000, how much less should each group get when we split 996 into 4 equal groups?" \
                    "3. Can we write (996) as (1000 - 4)? Then what would ((1000 - 4) div 4) become using the distributive property?" \
                    "4. What is (1000 div 4), and what is (4 div 4)? So what should ((1000 div 4) - (4 div 4)) be?" \
                    "5. When you added (9 + 9 + 6), did you keep the place values of 996, where the first 9 means 900 and the second 9 means 90?" \
                    "6. Is (996) the same number as (9 + 9 + 6), or is (996 = 900 + 90 + 6)? How does that change the division?" \
                    "7. If (24 div 4 = 6), what number are we dividing there: 24 or 996? Are those the same amount?" \
                    "8. Could adding the digits tell us something about divisibility, but not the actual answer to (996 div 4)? How could we check the actual quotient?" \
                    "9. If each group had 249, what would (249 times 4) give? How does that help check (996 div 4)?" \
                    "10. Can we split (996) into parts that are easy to divide by 4, like (800 + 196) or (1000 - 4), and divide each part correctly?"

  if (CONNECT_TO_CHATGPT_API):  
    response = client.responses.create(model="gpt-5.5",
                                       instructions="You are a math teacher assistant.",
                                       input=leading_question_prompt
                                      )
    response_output = response.output_text

  chatgpt_feedback_list = re.split(r'[0-9]+\.[ \t]+', response_output)
  if (len(chatgpt_feedback_list) > 0):
    chatgpt_feedback_list = chatgpt_feedback_list[1:]

  best_chatgpt_feedback = pick_best_chatgpt_feedback(problem_df, student_reasoning, chatgpt_feedback_list, num_leading_questions_to_supply)

  if (DEBUG_LEVEL > 0):
    print("chatgpt_feedback_list:  %s" % chatgpt_feedback_list)
    print("best_chatgpt_feedback:  %s" % best_chatgpt_feedback)

  return best_chatgpt_feedback


In [12]:
def justify_current_problem_solution_with_answers_to_previous_problems(math_problems):
    return_string = ""
    for math_problem in math_problems[:len(math_problems)-1]:
      return_string += math_problem['problem'] + ' = ' + math_problem['answer'] + ', '

    return return_string

In [13]:
import io
import json
import numpy as np

def evaluate_one_number_talk_string(number_talk_string_df, chatgpt_output_correlation_vector, perfect_correlation_vector):
  previous_student_reasoning = []
    
  if (DEBUG_LEVEL > 0):
    print(number_talk_string_df)
    
  num_candidate_answers = number_talk_string_df.num_candidate_answers[0]
  retry_max_times = 2
  if (DEBUG_LEVEL < EVALUATION_DEBUG_LEVEL):
    retry_max_times = number_talk_string_df.retry_max_times[0]
  math_problems = number_talk_string_df.problems[0]

  last_math_problem_in_string = math_problems[len(math_problems)-1]

  if (DEBUG_LEVEL > 0):
    print("last_math_problem_in_string:  %s" % last_math_problem_in_string)

  # ---------------------------------------------------------------------------
  # The following fulfills bullet point 3(a) in the block comment above.
  # problem_df = pd.DataFrame.from_dict(math_problem)
  problem_df = pd.DataFrame({key: pd.Series(value) for key, value in last_math_problem_in_string.items()})

  correct_answer_from_teacher = problem_df['answer'][0]

  if (DEBUG_LEVEL > 0):
    print("correct_answer_from_teacher:  %s" % correct_answer_from_teacher)
      
  answer_good_enough = False
  student_tries = 0
  while (not answer_good_enough and student_tries < retry_max_times):
    if ('instructions' in number_talk_string_df.keys()):
        instructions = number_talk_string_df.instructions[0];
        print(instructions)

    student_answer = VAGUE_WRONG_ANSWER
    if (CONTROL_RUN):
      student_answer = "Something around like " + correct_answer_from_teacher + " but not exactly ?"

    if (DEBUG_LEVEL < EVALUATION_DEBUG_LEVEL):
      student_answer = input("What is %s ?" % problem_df['problem'][0])

    # ---------------------------------------------------------------------------
    # The following fulfills bullet point 3(c) in the block comment above.
    if (student_tries == 0 and not CONTROL_RUN):
      student_reasoning = VAGUE_WRONG_REASONING
    else:
      student_reasoning = str(justify_current_problem_solution_with_answers_to_previous_problems(math_problems))
      if ('related_properties' in problem_df.keys() and len(problem_df['related_properties']) > 0):
        student_reasoning += " and " + str(problem_df['related_properties'][0]['property'] + " is relevant.")

    if (DEBUG_LEVEL < EVALUATION_DEBUG_LEVEL):
      student_reasoning = input("And what is your reasoning for your answer?")

    if (DEBUG_LEVEL > 0):
      print("Problem:  %s, Answer:  %s, Student Answer:  %s, Student Reasoning:  %s" \
            % (problem_df['problem'][0], correct_answer_from_teacher, student_answer, student_reasoning))

    student_answer_is_correct = False

    if (correct_answer_from_teacher == student_answer):
      student_answer_is_correct = True

    if (not student_answer_is_correct):
      # ---------------------------------------------------------------------------
      # The following fulfills bullet point 3(b) in the block comment above.
      verification_query = "A student answered that %s = %s.  Verify the answer is incorrect, yes or no." % (problem_df['problem'][0], student_answer)

      if (DEBUG_LEVEL > 1):
        print(verification_query)

      if (CONNECT_TO_CHATGPT_API):  
        response = client.responses.create(
          model="gpt-5.5",
          instructions="You are a math teacher assistant.",
          input=verification_query
        )

        if (response.output_text[:3].lower() != "yes"):
          student_answer_is_correct = True

    previous_student_reasoning.append(student_reasoning)
    student_tries += 1

    text_attributes_from_problem = get_text_attributes_from_problem(problem_df)

    if (not student_answer_is_correct):
      leading_question_from_chatgpt = chatgpt_api_supplies_leading_question(problem_df, student_answer, student_reasoning, num_candidate_answers)
      print(leading_question_from_chatgpt)

      if (DEBUG_LEVEL >= EVALUATION_DEBUG_LEVEL):
        chatgpt_output_correlation_to_student_reasoning = ask_gemini_to_correlate_phrases_in_a_0_to_10_ranking(student_reasoning, leading_question_from_chatgpt)
        chatgpt_output_correlation_to_correct_answer = ask_gemini_to_correlate_phrases_in_a_0_to_10_ranking(correct_answer_from_teacher, leading_question_from_chatgpt)  
        chatgpt_output_correlation_to_related_properties = ask_gemini_to_correlate_phrases_in_a_0_to_10_ranking(text_attributes_from_problem, leading_question_from_chatgpt)  

        perfect_correlation_vector = perfect_correlation_vector + np.array([1, 1, 1])
        chatgpt_output_correlation_vector = chatgpt_output_correlation_vector + \
                                              np.array([chatgpt_output_correlation_to_student_reasoning/10.0, chatgpt_output_correlation_to_correct_answer/10.0, chatgpt_output_correlation_to_related_properties/10.0])

        perfect_correlation_distance = vector_tip_distance(np.array([1, 1, 1]), np.array([0, 0, 0]))

        chatgpt_output_correlation_distance = perfect_correlation_distance - \
                                              vector_tip_distance(np.array([chatgpt_output_correlation_to_student_reasoning/10.0, chatgpt_output_correlation_to_correct_answer/10.0, chatgpt_output_correlation_to_related_properties/10.0]), \
                                                                  np.array([1, 1, 1]))

        print("For this one problem, the distance between the tip of the chatgpt_output_correlation_overall vector and the perfect_correlation_vector is %f" % chatgpt_output_correlation_distance)
        print("For this one problem %s, student_answer %s, text_attributes %s, and chatgpt output %s, this is a performance score of %s %%" \
              % (last_math_problem_in_string, student_reasoning, text_attributes_from_problem, leading_question_from_chatgpt, (chatgpt_output_correlation_distance/perfect_correlation_distance)*100.0))

      print("Let's try to think of another answer to this problem.")
      if (student_reasoning_is_relevant_to_problem(text_attributes_from_problem, student_reasoning, previous_student_reasoning)):
        print("Note: The reasoning you have provided was actually already along the lines of what the teacher was looking for.")
    else:
      print("Your answer was correct!")
      answer_good_enough = True

      reasoning_good_enough = False
      while (not reasoning_good_enough and student_tries < retry_max_times):    
        if (student_reasoning_is_relevant_to_problem(text_attributes_from_problem, student_reasoning, previous_student_reasoning)):
          print("Importantly, your reasoning was along the lines of what the teacher was looking for.")
          reasoning_good_enough = True
        else:
          if (DEBUG_LEVEL < EVALUATION_DEBUG_LEVEL):
            student_reasoning = input("Please add other reasoning for your answer to this problem.")
            previous_student_reasoning.append(student_reasoning)
          student_tries += 1

    if (DEBUG_LEVEL > 0):
      print("previous_student_reasoning:  %s" % previous_student_reasoning)

  return (chatgpt_output_correlation_vector, perfect_correlation_vector)


In [14]:
def vector_tip_distance(vector0, vector1):
  return np.sqrt((vector1[0] - vector0[0])**2 + \
                 (vector1[1] - vector0[1])**2 + \
                 (vector1[2] - vector0[2])**2)


In [15]:
from pathlib import Path
import pandas as pd

if (DEBUG_LEVEL < EVALUATION_DEBUG_LEVEL):
  problem_file_path = Path.cwd().glob("*.json")[0]
  print("File path:  %s" % posixPath)
  math_string_df = pd.read_json(posixPath)
  evaluate_one_number_talk_string(math_string_df, chatgpt_output_correlation_vector, perfect_correlation_vector)
  return

chatgpt_output_correlation_vector = np.array([0, 0, 0])
perfect_correlation_vector = np.array([0, 0, 0])

for posixPath in Path.cwd().glob("*.json"):
  print("File path:  %s" % posixPath)
  math_string_df = pd.read_json(posixPath)

  if (DEBUG_LEVEL > 0):
    print(math_string_df.head())

  (chatgpt_output_correlation_vector, perfect_correlation_vector) = \
    evaluate_one_number_talk_string(math_string_df, chatgpt_output_correlation_vector, perfect_correlation_vector)

perfect_correlation_distance = vector_tip_distance(perfect_correlation_vector, np.array([0, 0, 0]))

chatgpt_output_correlation_distance = perfect_correlation_distance - \
                                        vector_tip_distance(chatgpt_output_correlation_vector, \
                                                            perfect_correlation_vector)

print("The distance between the tip of the chatgpt_output_correlation_overall vector and the perfect_correlation_vector is %f" % chatgpt_output_correlation_distance)
if (perfect_correlation_distance != 0):
  print("This is a performance score of %s %%" % ((chatgpt_output_correlation_distance/perfect_correlation_distance)*100.0))
else:
  print("This is a performance score of 100 %%")


File path:  /home/myang/education/capstone-project/evaluation/MYangAICapstoneProjectUMGCForEvaluationString9.json
   num_candidate_answers  retry_max_times  \
0                      2                2   

                                            problems  
0  [{'problem': '5 * 20', 'answer': '100'}, {'pro...  
   num_candidate_answers  retry_max_times  \
0                      2                2   

                                            problems  
0  [{'problem': '5 * 20', 'answer': '100'}, {'pro...  
last_math_problem_in_string:  {'problem': '990 / 20', 'answer': '49.5', 'related_properties': [{'property': 'distributive', 'usage': '990 / 20 = (1000 - 10) / 20 = (1000 / 20) - (10/20) = 250 - 0.5 = 49.5'}]}
correct_answer_from_teacher:  49.5
Problem:  990 / 20, Answer:  49.5, Student Answer:  dunno, Student Reasoning:  dunno
A student answered that 990 / 20 = dunno.  Verify the answer is incorrect, yes or no.
Question giving to ChatGPT:  A child incorrectly thinks that 990 / 20

In [16]:
# 4.  Say "thank you for the conversation" to the student and assure them that good elements of the math conversation
#     have been recorded (logged) and will be passed to a human teacher for review.  Allow the student to sign off the session.
#
print("Thank you for having this math conversation!  Your responses will be reviewed by human teachers to help you to improve.")


Thank you for having this math conversation!  Your responses will be reviewed by human teachers to help you to improve.


In [17]:
# 5.  Disconnect from the ChatGPT API.
if (CONNECT_TO_CHATGPT_API):
  client.close()
